# Zero-Shot Pre-Labeling for Negative and Positive Reviews

This notebook uses a zero-shot classification model to pre-label PickMe reviews.

We create two separate labeled datasets:

1. Negative reviews → complaint categories
2. Positive reviews → satisfaction categories

These pre-labeled datasets will later be manually reviewed and corrected.

In [16]:
import pandas as pd
from pathlib import Path
from transformers import pipeline
from tqdm import tqdm
import torch
import re

In [22]:
RAW_PATH = Path("../data/raw/pickme_reviews_with_sentiment.csv")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

Load, Clean, and Filter Noise

In [24]:
# Load data
df = pd.read_csv(RAW_PATH)

# Keep English reviews only
df_en = df[df["language"].astype(str).str.strip().str.lower().isin(["english", "en"])].copy()

# Clean review text
df_en["review_text"] = df_en["review_text"].astype(str).str.strip()

# Remove empty reviews
df_en = df_en[df_en["review_text"].str.len() > 0].copy()

# Remove very short reviews
# Short reviews usually do not contain enough information for topic classification
df_en = df_en[df_en["review_text"].str.len() >= 15].copy()

# Clean sentiment labels safely
df_en["sentiment_clean"] = (
    df_en["sentiment"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(
        lambda x: "NEGATIVE" if "neg" in x
        else "POSITIVE" if "pos" in x
        else "OTHER"
    )
)

# Keep only Negative and Positive
# This removes "Not Analyzed"
df_en = df_en[df_en["sentiment_clean"].isin(["NEGATIVE", "POSITIVE"])].copy()

# Split into negative and positive
df_neg = df_en[df_en["sentiment_clean"] == "NEGATIVE"].copy().reset_index(drop=True)
df_pos = df_en[df_en["sentiment_clean"] == "POSITIVE"].copy().reset_index(drop=True)

print(f"Negative reviews after cleaning: {len(df_neg)}")
print(f"Positive reviews after cleaning: {len(df_pos)}")

Negative reviews after cleaning: 745
Positive reviews after cleaning: 349


Define category labels

In [25]:
NEGATIVE_LABELS = [
    "app bugs, crashes, slow app, technical issues, map, GPS, login, payment failure",
    "pricing, surge pricing, bidding, driver offers, high fares, expensive, discounts",
    "food delivery delays, late orders, missing items, wrong orders, cold food",
    "driver behavior, rude driver, unsafe driving, driver no show, safety issues",
    "customer support, refunds, complaint handling, unhelpful support",
    "privacy, security, data, account safety"
]

POSITIVE_LABELS = [
    "app easy to use, fast app, user friendly, app works well",
    "fair pricing, cheap fares, transparent pricing, discounts, good value",
    "fast delivery, on time delivery, reliable delivery, food arrived hot",
    "polite driver, safe driving, professional driver, friendly driver",
    "helpful customer support, quick refund, good service",
    "privacy, secure payments, account safety, trust"
]

Load the zero-shot model

In [26]:
print("Loading model...")
classifier = pipeline(
    "zero-shot-classification", 
    model="facebook/bart-large-mnli", 
    device=0
)

Loading model...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Classification function

In [27]:
def pre_label_reviews(df, labels, batch_size=8):
    df = df.copy()
    texts = df["review_text"].tolist()

    predicted_labels = []
    confidence_scores = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i : i + batch_size]

        batch_results = classifier(batch_texts, labels)

        for result in batch_results:
            predicted_labels.append(result["labels"][0])
            confidence_scores.append(result["scores"][0])

    df["predicted_category"] = predicted_labels
    df["category_confidence"] = confidence_scores

    return df

Pre-label negative reviews

In [28]:
df_neg_labeled = pre_label_reviews(df_neg, NEGATIVE_LABELS, batch_size=8)

100%|██████████| 94/94 [19:53<00:00, 12.69s/it]


Pre-label positive reviews

In [29]:
df_pos_labeled = pre_label_reviews(df_pos, POSITIVE_LABELS, batch_size=8)

100%|██████████| 44/44 [05:23<00:00,  7.36s/it]


Check quick results

In [31]:
print("--- Negative Categories ---")
print(df_neg_labeled["predicted_category"].value_counts())

print("\n--- Positive Categories ---")
print(df_pos_labeled["predicted_category"].value_counts())

print("\nNegative average confidence:")
print(df_neg_labeled["category_confidence"].mean())

print("\nPositive average confidence:")
print(df_pos_labeled["category_confidence"].mean())

--- Negative Categories ---
predicted_category
driver behavior, rude driver, unsafe driving, driver no show, safety issues         288
customer support, refunds, complaint handling, unhelpful support                    131
app bugs, crashes, slow app, technical issues, map, GPS, login, payment failure     117
privacy, security, data, account safety                                              86
pricing, surge pricing, bidding, driver offers, high fares, expensive, discounts     71
food delivery delays, late orders, missing items, wrong orders, cold food            52
Name: count, dtype: int64

--- Positive Categories ---
predicted_category
polite driver, safe driving, professional driver, friendly driver        160
app easy to use, fast app, user friendly, app works well                 146
fair pricing, cheap fares, transparent pricing, discounts, good value     25
helpful customer support, quick refund, good service                      14
privacy, secure payments, account safety, t

Map long labels to short labels


In [32]:
NEG_LABEL_MAP = {
    "app bugs, crashes, slow app, technical issues, map, GPS, login, payment failure": "App Bugs",
    "pricing, surge pricing, bidding, driver offers, high fares, expensive, discounts": "Pricing & Bidding",
    "food delivery delays, late orders, missing items, wrong orders, cold food": "Delivery Issues",
    "driver behavior, rude driver, unsafe driving, driver no show, safety issues": "Driver Behavior & Safety",
    "customer support, refunds, complaint handling, unhelpful support": "Support & Refunds",
    "privacy, security, data, account safety": "Privacy & Security"
}

POS_LABEL_MAP = {
    "app easy to use, fast app, user friendly, app works well": "App Experience",
    "fair pricing, cheap fares, transparent pricing, discounts, good value": "Fair Pricing",
    "fast delivery, on time delivery, reliable delivery, food arrived hot": "Fast Delivery",
    "polite driver, safe driving, professional driver, friendly driver": "Driver Service",
    "helpful customer support, quick refund, good service": "Good Support",
    "privacy, secure payments, account safety, trust": "Trust & Security"
}

df_neg_labeled["category"] = df_neg_labeled["predicted_category"].map(NEG_LABEL_MAP)
df_pos_labeled["category"] = df_pos_labeled["predicted_category"].map(POS_LABEL_MAP)

# Check that no labels failed to map
print("Negative missing mapped labels:", df_neg_labeled["category"].isna().sum())
print("Positive missing mapped labels:", df_pos_labeled["category"].isna().sum())

Negative missing mapped labels: 0
Positive missing mapped labels: 0


Add confidence bands

In [33]:
def confidence_band(score):
    if score >= 0.65:
        return "high"
    elif score >= 0.45:
        return "medium"
    else:
        return "low"

df_neg_labeled["confidence_band"] = df_neg_labeled["category_confidence"].apply(confidence_band)
df_pos_labeled["confidence_band"] = df_pos_labeled["category_confidence"].apply(confidence_band)

print("--- Negative Confidence Bands ---")
print(df_neg_labeled["confidence_band"].value_counts())

print("\n--- Positive Confidence Bands ---")
print(df_pos_labeled["confidence_band"].value_counts())

--- Negative Confidence Bands ---
confidence_band
low       345
medium    203
high      197
Name: count, dtype: int64

--- Positive Confidence Bands ---
confidence_band
low       128
medium    127
high       94
Name: count, dtype: int64


Add review length and label source

In [34]:
df_neg_labeled["review_length"] = df_neg_labeled["review_text"].str.len()
df_pos_labeled["review_length"] = df_pos_labeled["review_text"].str.len()

df_neg_labeled["label_source"] = "zero-shot-bart-large-mnli"
df_pos_labeled["label_source"] = "zero-shot-bart-large-mnli"

Save the pre-labeled datasets

In [35]:
df_neg_labeled.to_csv(PROCESSED_DIR / "negative_pre_labeled.csv", index=False)
df_pos_labeled.to_csv(PROCESSED_DIR / "positive_pre_labeled.csv", index=False)

print("Saved: data/processed/negative_pre_labeled.csv")
print("Saved: data/processed/positive_pre_labeled.csv")

Saved: data/processed/negative_pre_labeled.csv
Saved: data/processed/positive_pre_labeled.csv


Quick check

In [36]:
print(df_neg_labeled[["review_text", "category", "category_confidence", "confidence_band"]].head())
print(df_pos_labeled[["review_text", "category", "category_confidence", "confidence_band"]].head())

                                                                                                                                             review_text  \
0          these guys are terrible, they dont take any responsibility. customer service ignores you and closes tickets. absolute worst customer service.   
1  this happened to me multiple times where i order food and it takes hours for a rider to get assigned. there should be an option to cancel the orde...   
2                                                                    this app is a not trust that is laying 50% discunt frist ride. not it offer working   
3  The new “Bid” feature has made PickMe extremely frustrating. Bidding gets activated automatically, and drivers take a long time to accept rides be...   
4                                                                                                                                        cheap and quick   

                   category  category_confidence confidence_ban